## Optimizacion combinatoria

In [1]:
#! pip install cvxpy
import cvxpy as cp
import numpy as np

### Ejemplo 0


In [25]:
import cvxpy as cp
import numpy as np

x = cp.Variable(shape=(2,1), name="x")

objective = cp.Maximize(4*x[0] + 5*x[1])

# Relajación lineal continua
constraints = [
    4*x[0] + 6*x[1] <= 24,
    2*x[0] + x[1] <= 6,
    x[0] - x[1] <= 1,
    x[0] <= 2,
    x[0] <= 1, # Ramificar x_1
    x[1] >= 4, # Ramificar x_2
    x >= 0
]

In [26]:
problem = cp.Problem(objective, constraints)
solution = problem.solve()

print("Valor máximo:", np.round(solution, 4))
print("Solución:\n", np.round(x.value, 4))

Valor máximo: 20.0
Solución:
 [[0.]
 [4.]]


Tenemos un elemento que no es entero: $x_1$  
$\Rightarrow$ Ocupamos ramificar, añadiendo dos modelos distintos con las restricciones $x_1\leq 1$ y $x_1\geq 2$.  

* Con $x_1 \geq 2$ resulta en la solución $x^T = (2,2)$ y $z^* = 18$

* Con $x_1 \leq 1$ resulta en la solución $x^T = (1,3.333)$ y $z^* = 20.66$
    - Se obtiene una solución no entera para $x_2 \Rightarrow$ volver a ramificar con $x_2<=3$ y $x_2>=4$ 
        1. $x^T = (1,3)$ y $z^* = 19$
        2. $x^T = (0,4)$ y $z^* = 20$


Por lo tanto, nos quedamos con la solución  $x^T = (0,4)$ y $z^* = 20$


In [28]:
# Forzando enteros
x = cp.Variable(shape=(2,1), name="x", integer=True)

objective = cp.Maximize(4*x[0] + 5*x[1])

# Relajación lineal continua
constraints = [
    4*x[0] + 6*x[1] <= 24,
    2*x[0] + x[1] <= 6,
    x[0] - x[1] <= 1,
    x[0] <= 2,
    x >= 0
]
problem = cp.Problem(objective, constraints)
solution = problem.solve()

print("Valor máximo:", np.round(solution, 4))
print("Solución:\n", np.round(x.value, 4))

Valor máximo: 20.0
Solución:
 [[-0.]
 [ 4.]]


### Ejemplo

In [42]:
x = cp.Variable(shape=(2,1), name="x", integer=True)

objective = cp.Maximize(8*x[0] + 5*x[1])

# Relajación lineal continua
constraints = [
    1*x[0] + 1*x[1] <= 6,
    9*x[0] + 5*x[1] <= 45,
    x >= 0
]
problem = cp.Problem(objective, constraints)
solution = problem.solve()

print("Valor máximo:", np.round(solution, 4))
print("Solución:\n", np.round(x.value, 4))

Valor máximo: 40.0
Solución:
 [[ 5.]
 [-0.]]


### Otro ejemplote

In [2]:
# Relajación lineal continua
x = cp.Variable(shape=(2,1), name="x")

objective = cp.Maximize(2*x[0] + 3*x[1])

constraints = [
    -3*x[0] + x[1] <= 1,
    4*x[0] + 2*x[1] <= 15,
    4*x[0] - x[1] <= 10,
    -x[0] + 2*x[1] <= 5,
    x >= 0
]

In [ ]:
problem = cp.Problem(objective, constraints)
solution = problem.solve()
print("Valor máximo:", np.round(solution, 4))
print("Solución:\n", np.round(x.value, 4))

Valor máximo: 14.5
Solución:
 [[2. ]
 [3.5]]


**Relajación Lineal:** $z^* = 14.5$, $x^* = (2,3.5)^T$  
Entonces, ahora probamos ramificando en $x_2$: $x_2 \leq 3$ y $x_2 \geq 4$

In [7]:
# x_2 <= 3
print("x_2 <= 3")
constraints = [
    -3*x[0] + x[1] <= 1,
    4*x[0] + 2*x[1] <= 15,
    4*x[0] - x[1] <= 10,
    -x[0] + 2*x[1] <= 5,
    x[1] <= 3, # Ramificar x_2
    x >= 0
]
problem = cp.Problem(objective, constraints)
solution = problem.solve()
print("Valor máximo:", np.round(solution, 4))
print("Solución:\n", np.round(x.value, 4))

# Ahora x_2 <= 3 y x_1 <= 2
print("x_2 <= 3 y x_1 <= 2")
constraints = [
    -3*x[0] + x[1] <= 1,
    4*x[0] + 2*x[1] <= 15,
    4*x[0] - x[1] <= 10,
    -x[0] + 2*x[1] <= 5,
    x[1] <= 3, # Ramificar x_2
    x[0] <= 2, # Ramificar x_1
    x >= 0
]
problem = cp.Problem(objective, constraints)
solution = problem.solve()
print("Valor máximo:", np.round(solution, 4))
print("Solución:\n", np.round(x.value, 4))

# Ahora x_2 <= 3 y x_1 >= 3
# NO jala


x_2 <= 3
Valor máximo: 13.5
Solución:
 [[2.25]
 [3.  ]]
x_2 <= 3 y x_1 <= 2
Valor máximo: 13.0
Solución:
 [[2.]
 [3.]]


In [9]:
# x_2 >= 4
# NO JALA

$\therefore z^* = 13$ y $x^* = (2,3)^T$

### Tarea

In [ ]:
def branch_and_bound(x, objective, constraints : list, title = "Inicio", nivel = 0):
    
    problem = cp.Problem(objective, constraints)
    solution = problem.solve()
    
    if solution is None:
        return None
    
    print(title)
    print(f"Valor máximo: {solution}\nSolución:\n {x.value}\n")
    
    if x.value is None:
        return None
    
    # Verificar si la solución es entera
    x_val = np.round(x.value.flatten(),2)
    if np.all(np.isclose(x_val, np.round(x_val))):
        return None

    nivel += 1
    
    if not np.isclose(float(x_val[0]) % 1, 0):
        # Ramificar
        
        # Ramificar x_1
        inferior = np.floor(x_val[0])
        superior = np.ceil(x_val[0])
        constraints_1 = constraints.copy()
        
        constraints_1.append(x[0] <= inferior)
        title = "Ramificando x_1 <= " + str(inferior) + " nivel " + str(nivel)
        branch_and_bound(x, objective, constraints_1, title, nivel)
        constraints_1.pop()
        
        constraints_1.append(x[0] >= superior)
        title = "Ramificando x_1 >= " + str(superior) + " nivel " + str(nivel)
        branch_and_bound(x, objective, constraints_1, title, nivel)
        
    if not np.isclose(float(x_val[1]) % 1, 0):
        
        # Ramificar x_2
        inferior = np.floor(x_val[1])
        superior = np.ceil(x_val[1])
        constraints_2 = constraints.copy()
        
        constraints_2.append(x[1] <= inferior)
        title = "Ramificando x_2 <= " + str(inferior) + " nivel " + str(nivel)
        branch_and_bound(x, objective, constraints_2, title, nivel)
        constraints_2.pop()
        
        constraints_2.append(x[1] >= superior)
        title = "Ramificando x_2 >= " + str(superior) + " nivel " + str(nivel)
        branch_and_bound(x, objective, constraints_2, title, nivel)
        
        return None

In [39]:
x = cp.Variable(shape=(2,1), name="x")

objective = cp.Maximize(80*x[0] + 45*x[1])

# Relajación lineal continua
constraints = [
    x[0] + x[1] <= 7,
    12*x[0] + 5*x[1] <= 60,
    x >= 0
]

branch_and_bound(x, objective, constraints)

Inicio
Valor máximo: 439.99999986105223
Solución:
 [[3.57142857]
 [3.42857142]]

Ramificando x_1 <= 3.0 nivel 1
Valor máximo: 419.99999999534845
Solución:
 [[3.]
 [4.]]

Ramificando x_1 >= 4.0 nivel 1
Valor máximo: 428.0000000768551
Solución:
 [[4. ]
 [2.4]]

Ramificando x_2 <= 2.0 nivel 2
Valor máximo: 423.33333354461774
Solución:
 [[4.16666667]
 [2.        ]]

Ramificando x_1 <= 4.0 nivel 3
Valor máximo: 410.00000003126576
Solución:
 [[4.]
 [2.]]

Ramificando x_1 >= 5.0 nivel 3
Valor máximo: 400.0000017943278
Solución:
 [[ 5.00000003e+00]
 [-8.11760221e-09]]

Ramificando x_2 >= 3.0 nivel 2
Valor máximo: -inf
Solución:
 None

Ramificando x_2 <= 3.0 nivel 1
Valor máximo: 434.9999999021502
Solución:
 [[3.75000001]
 [2.99999999]]

Ramificando x_1 <= 3.0 nivel 2
Valor máximo: 374.9999999919824
Solución:
 [[3.]
 [3.]]

Ramificando x_1 >= 4.0 nivel 2
Valor máximo: 428.00000007882
Solución:
 [[4. ]
 [2.4]]

Ramificando x_2 <= 2.0 nivel 3
Valor máximo: 423.3333335467664
Solución:
 [[4.1666666